# Reading GRIN2B line NWB Files

This notebook demonstrates how to read the converted GRIN2B NWB files from the Gonzalez-Sulser Lab. It covers:

1. Loading the NWB file and inspecting top-level metadata
2. Reading raw EEG/EMG acquisition data
3. Full-recording EEG spectrogram
4. Reading sleep-state epoch annotations
5. Sleep states by hour
6. Reading per-sleep-state power spectra
7. Reading seizure event intervals
8. EEG/EMG around a seizure event
9. Reading per-epoch spike-wave discharge (SWD) counts

**Prerequisites:** `pynwb`, `matplotlib`, `numpy`, `pandas`

```bash
conda activate gonzalez-sulser-lab-to-nwb-env
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from pynwb import NWBHDF5IO

# Update this path to the location of your NWB file.
NWB_PATH = Path(
    "H:/gonzalez-nwbfiles/sub-GRIN2B-129/sub-GRIN2B-129_ses-2021-03-26_ecephys.nwb"
)

## 1. Load and inspect top-level metadata

In [ ]:
io = NWBHDF5IO(NWB_PATH, mode="r")
nwb = io.read()
nwb.experiment_description

In [ ]:
print(f"Session ID:          {nwb.session_id}")
print(f"Session start time:  {nwb.session_start_time}")
print(f"Institution:         {nwb.institution}")
print(f"Lab:                 {nwb.lab}")
print(f"Subject ID:          {nwb.subject.subject_id}")
print(f"Subject species:     {nwb.subject.species}")
print()
print("Acquisition keys:", list(nwb.acquisition.keys()))
print("Processing modules:", list(nwb.processing.keys()))

epochs_df = nwb.epochs.to_dataframe()
print()
print(
    "Baseline windows 1 and 2 are marked in the NWB epochs table (seconds relative to session_start_time)"
)
epochs_df[:]
bl1_start_time = epochs_df["start_time"][0]
bl2_start_time = epochs_df["start_time"][1]

## 2. Raw EEG / EMG

The full 24 h recording is stored as **two** `ElectricalSeries` in `acquisition` - one for
EEG, one for EMG - sharing a single electrode table but linked to distinct `ElectrodeGroup`s:

- `EEGElectricalSeries{baseline_name}`: **14 EEG channels** - bilateral cortex - S1_Tr, M2_Fra, M2_anterior, M1_anterior, V2_ML, V1_M, S1Hl_S1Fl (Right + Left)
- `EMGElectricalSeries{baseline_name}`: **2 EMG channels** - Right and Left


In [ ]:
eeg_name = next(name for name in nwb.acquisition if name.startswith("EEGElectricalSeries"))
emg_name = next(name for name in nwb.acquisition if name.startswith("EMGElectricalSeries"))
eeg_es = nwb.acquisition[eeg_name]
emg_es = nwb.acquisition[emg_name]

for name, es in [("EEG", eeg_es), ("EMG", emg_es)]:
    print(f"--- {name} ---")
    print(f"ElectricalSeries shape:  {es.data.shape}")
    print(f"Sampling rate:           {es.rate} Hz")
    print(
        f"Starting time:           {es.starting_time:.2f} s (= session_start_time; full recording)"
    )
    print(f"Duration:                {es.data.shape[0] / es.rate / 3600:.2f} h")
    print(f"Conversion (to volts):   {es.conversion:.6e}")
    print()

electrodes = nwb.electrodes.to_dataframe()
electrodes[:]

In [ ]:
fs = eeg_es.rate
window_start_s = 15
window_duration_s = 30
window_end_s = window_start_s + window_duration_s

i_start = int(window_start_s * fs)
i_end = int(window_end_s * fs)

# Convert raw int16 ADC counts to volts using the confirmed TainiTec gain
# (es.conversion is in volts/count). The full recording is indexed directly
eeg_slice = eeg_es.data[i_start:i_end, :] * eeg_es.conversion  # lazy HDF5 read
emg_slice = emg_es.data[i_start:i_end, :] * emg_es.conversion  # lazy HDF5 read

t = eeg_es.get_timestamps()[i_start:i_end]

# EEG channels: indices within EEGElectricalSeries (14 channels) — pick a bilateral pair
eeg_idx = {"S1_Tr_R": 0, "M1_anterior_R": 3, "S1_Tr_L": 13, "M1_anterior_L": 10}
# EMG channels: indices within EMGElectricalSeries (2 channels)
emg_idx = {"EMG_R": 0, "EMG_L": 1}

fig, axes = plt.subplots(len(eeg_idx) + 1, 1, figsize=(14, 7), sharex=True)

for ax, (label, idx) in zip(axes[:-1], eeg_idx.items()):
    ax.plot(t, eeg_slice[:, idx], lw=0.4)
    ax.set_ylabel(f"{label}\n(V)", fontsize=8)

ax_emg = axes[-1]
for label, idx in emg_idx.items():
    ax_emg.plot(t, emg_slice[:, idx], lw=0.4, label=label)
ax_emg.set_ylabel("EMG (V)", fontsize=8)
ax_emg.legend(fontsize=7, loc="upper right")
ax_emg.set_xlabel(f"Time (s)")

axes[0].set_title(f"Raw EEG/EMG - subject {nwb.subject.subject_id}")
plt.tight_layout()
plt.show()

## 3. Full-recording EEG spectrogram

Reproduces the lab's per-epoch spectrogram plot (`spec_129_BL2.png`): raw EEG power is
computed in the same 5-s epochs as the sleep scoring, then split into fixed-duration
panels stacked top-to-bottom to cover the full recording.

In [ ]:
# ── Spectrogram parameters ────────────────────────────────────────────────────
channel_idx = 0                # EEG channel to use (index into eeg_es channels)
freq_min, freq_max = 0.4, 20.0  # Hz - matches the lab's plotting band
panel_duration_h = 2.0          # hours of recording per stacked panel
# ──────────────────────────────────────────────────────────────────────────────

fs = eeg_es.rate
epoch_len_s = 5.0  # same epoch length as the sleep scoring (Section 4)
n_per_epoch = int(round(epoch_len_s * fs))
n_epochs = eeg_es.data.shape[0] // n_per_epoch

eeg_channel = eeg_es.data[: n_epochs * n_per_epoch, channel_idx]  # lazy HDF5 read
epochs = eeg_channel.reshape(n_epochs, n_per_epoch)

freqs = np.fft.rfftfreq(n_per_epoch, d=1 / fs)
power = np.abs(np.fft.rfft(epochs, axis=1)) ** 2

freq_mask = (freqs >= freq_min) & (freqs <= freq_max)
freqs = freqs[freq_mask]
log_power = np.log10(power[:, freq_mask] + np.finfo(float).eps)

epochs_per_panel = int(panel_duration_h * 3600 / epoch_len_s)
n_panels = int(np.ceil(n_epochs / epochs_per_panel))
vmin, vmax = np.percentile(log_power, [1, 99])

fig, axes = plt.subplots(n_panels, 1, figsize=(9, 1.1 * n_panels), sharex=True)
for panel, ax in enumerate(axes):
    i0 = panel * epochs_per_panel
    i1 = min(i0 + epochs_per_panel, n_epochs)
    ax.imshow(
        log_power[i0:i1].T,
        aspect="auto",
        origin="lower",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        extent=[0, i1 - i0, freq_min, freq_max],
    )
    ax.set_yticks([])

axes[-1].set_xlabel("epochs (5s)")
fig.text(
    0.04, 0.5, f"freq, {freq_min}-{freq_max}Hz 0.2Hz res.",
    va="center", rotation="vertical",
)
fig.suptitle("Spectrogram")
plt.subplots_adjust(hspace=0.05, left=0.12)
plt.show()

## 4. Sleep-state epochs

5-second scored epochs in `processing/behavior/sleep_states_baseline_window_1` (BL1; BL2 is
`sleep_states_baseline_window_2`).
State codes: 0 = Wake, 1 = NREM, 2 = REM. State 4 = SWD

In [ ]:
behavior = nwb.processing["behavior"]
sleep_baseline_window_1 = behavior["sleep_states_baseline_window_1"]
sleep_df_bl1 = sleep_baseline_window_1.to_dataframe()
sleep_baseline_window_2 = behavior["sleep_states_baseline_window_2"]
sleep_df_bl2 = sleep_baseline_window_2.to_dataframe()

In [ ]:
print(
    f"Sleep epochs in the baseline window 1: {len(sleep_df_bl1)} rows (expected 17,280 for full 24 h)"
)
sleep_df_bl1[:]

In [ ]:
print(
    f"Sleep epochs in the baseline window 2: {len(sleep_df_bl2)} rows (expected 17,280 for full 24 h)"
)
sleep_df_bl2[:]

In [ ]:
# Sleep hypnogram
state_labels = {0: "Wake", 1: "NREM", 2: "REM", 4: "SWD"}
t_mid = (sleep_df_bl1["start_time"] + sleep_df_bl1["stop_time"]) / 2
hours = (t_mid - bl1_start_time) / 3600  # hours into BL1 window

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].scatter(
    hours,
    sleep_df_bl1["sleep_score"],
    s=3,
    c=sleep_df_bl1["sleep_score"],
    cmap="tab10",
    vmin=0,
    vmax=4,
)
axes[0].set_yticks([0, 1, 2])
axes[0].set_yticklabels(["Wake", "NREM", "REM"])
axes[0].set_xlabel("Hours into baseline window 1")
axes[1].scatter(
    hours,
    sleep_df_bl2["sleep_score"],
    s=3,
    c=sleep_df_bl2["sleep_score"],
    cmap="tab10",
    vmin=0,
    vmax=4,
)
axes[1].set_yticks([0, 1, 2])
axes[1].set_yticklabels(["Wake", "NREM", "REM"])
axes[1].set_xlabel("Hours into baseline window 2")
axes[0].set_title(f"Sleep hypnogram - subject {nwb.subject.subject_id}")
plt.tight_layout()
plt.show()

## 5. Sleep states by hour

Stacked bar chart of epoch counts per hour since lights-on, reproducing the lab's
`States by Hour` plot (e.g. `sbh_129_BL1.png`).

In [ ]:
state_names = {0: "WAKE", 1: "NonREM", 2: "REM"}

# Bin each 5-s epoch into the hour (since lights-on / BL window start) it falls in
hours_since_lights_on_bl1 = np.floor(
    (sleep_df_bl1["start_time"] - eeg_es.starting_time) / 3600
).astype(int)

counts_by_hour_bl1 = (
    sleep_df_bl1["sleep_score"]
    .map(state_names)
    .groupby(hours_since_lights_on_bl1)
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=["REM", "NonREM", "WAKE"], fill_value=0)
    .sort_index()
)

hours_since_lights_on_bl2 = np.floor(
    (sleep_df_bl2["start_time"] - eeg_es.starting_time) / 3600
).astype(int)

counts_by_hour_bl2 = (
    sleep_df_bl2["sleep_score"]
    .map(state_names)
    .groupby(hours_since_lights_on_bl2)
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=["REM", "NonREM", "WAKE"], fill_value=0)
    .sort_index()
)

fig, axes = plt.subplots(1,2, figsize=(9, 6))
bottom = np.zeros(len(counts_by_hour_bl1))
colors = {"WAKE": "limegreen", "NonREM": "blue", "REM": "red"}
for state in ["REM", "NonREM", "WAKE"]:
    axes[0].bar(
        counts_by_hour_bl1.index,
        counts_by_hour_bl1[state],
        bottom=bottom,
        width=0.9,
        color=colors[state],
        label=state,
    )
    bottom += counts_by_hour_bl1[state].to_numpy()

axes[0].set_xlabel("Hours since lights on")
axes[0].set_ylabel("epochs")
axes[0].set_title("States by Hour - baseline window 1")

bottom = np.zeros(len(counts_by_hour_bl2))
colors = {"WAKE": "limegreen", "NonREM": "blue", "REM": "red"}
for state in ["REM", "NonREM", "WAKE"]:
    axes[1].bar(
        counts_by_hour_bl2.index,
        counts_by_hour_bl2[state],
        bottom=bottom,
        width=0.9,
        color=colors[state],
        label=state,
    )
    bottom += counts_by_hour_bl2[state].to_numpy()

axes[1].set_xlabel("Hours since lights on")
axes[1].set_ylabel("epochs")
axes[1].set_title("States by Hour - baseline window 2")
axes[1].legend(title="States", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 6. Per-sleep-state power spectra

628 frequency bins (0→125.2 Hz, 0.2 Hz steps = Nyquist of 250.4 Hz).
Columns are named after the brain state: `Wake`, `NREM`, `REM`, `SWD`

In [ ]:
ecephys = nwb.processing["ecephys"]
psd_table_bl1 = ecephys["sleep_state_power_spectra_baseline_window_1"]
psd_df_bl1 = psd_table_bl1.to_dataframe()
psd_table_bl2 = ecephys["sleep_state_power_spectra_baseline_window_2"]
psd_df_bl2 = psd_table_bl2.to_dataframe()

In [ ]:
print(f"PSD table: {psd_df_bl1.shape}")
psd_df_bl1[:]

In [ ]:
print(f"PSD table: {psd_df_bl2.shape}")
psd_df_bl2[:]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
state_cols = ["Wake", "NREM", "REM", "SWD"]
for col in state_cols:
    if col in psd_df_bl1.columns:
        axes[0].semilogy(psd_df_bl1["frequency_hz"], psd_df_bl1[col], label=col)
# ax.set_xlim(0, 50)
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Power (a.u.)")
axes[0].set_title("Sleep-state power spectra -baseline window 1")
# axes[0].legend()
for col in state_cols:
    if col in psd_df_bl2.columns:
        axes[1].semilogy(psd_df_bl2["frequency_hz"], psd_df_bl2[col], label=col)
# ax.set_xlim(0, 50)
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_title("Sleep-state power spectra - baseline window 2")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Seizure events

Stored as a `pynwb.epoch.TimeIntervals` table in `processing/behavior/seizure_eventss_baseline_window_1`/`_2`.
The table has one row per seizure event, with `start_time`, `stop_time`, and `duration` columns.

In [ ]:
seizure_table = behavior["seizure_events_baseline_window_1"]
seizure_df = seizure_table.to_dataframe()

start_times = seizure_df["start_time"].to_numpy()
stop_times = seizure_df["stop_time"].to_numpy()
durations = seizure_df["duration"].to_numpy()

fig, axes = plt.subplots(2, 2, figsize=(14, 4), sharex="col")

# Duration histogram
axes[0][0].hist(durations, bins=20, color="steelblue", edgecolor="white")
axes[0][0].set_xlabel("Duration (s)")
axes[0][0].set_ylabel("Count")
axes[0][0].set_title(f"Seizure duration distribution (n={len(durations)})")

# Event timeline relative to BL1 window start
rel_start = (start_times - bl1_start_time) / 3600
axes[0][1].eventplot(rel_start, lineoffsets=0, linelengths=0.5, color="firebrick")
axes[0][1].set_xlabel("Hours into baseline window 1")
axes[0][1].set_yticks([])
axes[0][1].set_title("Seizure onset times over 24 h (BL1)")

seizure_table = behavior["seizure_events_baseline_window_2"]
seizure_df = seizure_table.to_dataframe()

start_times = seizure_df["start_time"].to_numpy()
stop_times = seizure_df["stop_time"].to_numpy()
durations = seizure_df["duration"].to_numpy()

# Duration histogram
axes[1][0].hist(durations, bins=20, color="steelblue", edgecolor="white")
axes[1][0].set_xlabel("Duration (s)")
axes[1][0].set_ylabel("Count")
axes[1][0].set_title(f"Seizure duration distribution (n={len(durations)})")

# Event timeline relative to BL1 window start
rel_start = (start_times - bl2_start_time) / 3600
axes[1][1].eventplot(rel_start, lineoffsets=0, linelengths=0.5, color="firebrick")
axes[1][1].set_xlabel("Hours into baseline window 2")
axes[1][1].set_yticks([])
axes[1][1].set_title("Seizure onset times over 24 h (BL2)")

plt.tight_layout()
plt.show()

## 8. EEG/EMG around a seizure event

Pick a seizure by index (`seizure_idx`) from the `seizure_events` table read in Section 7,
and automatically extract +/- `pad_s` seconds of raw EEG/EMG around it for plotting.

In [ ]:
# ── Seizure selection ───────────────────────────────────────────────────────
seizure_idx = 3  # 10th seizure event (0-indexed)
pad_s = 1.0      # seconds of padding before/after the seizure to include
# ─────────────────────────────────────────────────────────────────────────────

seizure_onset_s = start_times[seizure_idx]   # absolute time (s), from Section 7
seizure_offset_s = stop_times[seizure_idx]

fs = eeg_es.rate

window_start_s = seizure_onset_s - pad_s
window_end_s = seizure_offset_s + pad_s

i_start = int(window_start_s * fs)
i_end = int(window_end_s * fs)
t = np.arange(i_start, i_end) / fs - (seizure_onset_s)  # t=0 at seizure onset

eeg_slice = eeg_es.data[i_start:i_end, :]  # lazy HDF5 read
emg_slice = emg_es.data[i_start:i_end, :]  # lazy HDF5 read

eeg_idx = {"S1_Tr_R": 0, "M1_anterior_R": 3, "S1_Tr_L": 13, "M1_anterior_L": 10}
emg_idx = {"EMG_R": 0, "EMG_L": 1}

fig, axes = plt.subplots(len(eeg_idx) + 1, 1, figsize=(14, 7), sharex=True)

for ax, (label, idx) in zip(axes[:-1], eeg_idx.items()):
    ax.plot(t, eeg_slice[:, idx], lw=0.4)
    ax.set_ylabel(label, fontsize=8)

ax_emg = axes[-1]
for label, idx in emg_idx.items():
    ax_emg.plot(t, emg_slice[:, idx], lw=0.4, label=label)
ax_emg.set_ylabel("EMG", fontsize=8)
ax_emg.legend(fontsize=7, loc="upper right")
ax_emg.set_xlabel("Time (s, relative to seizure onset)")

seizure_duration_s = seizure_offset_s - seizure_onset_s
for ax in axes:
    ax.axvspan(0, seizure_duration_s, color="firebrick", alpha=0.15)

axes[0].set_title(
    f"EEG/EMG around seizure #{seizure_idx} "
    f"(duration {seizure_duration_s:.2f} s, +/-{pad_s}s padding) - GRIN2B_129"
)
plt.tight_layout()
plt.show()

## 9. Per-epoch SWD counts

`swd_epoch_counts` is a `TimeSeries` at 0.2 Hz (one value per 5-s sleep epoch).

In [ ]:
swd = behavior["swd_epoch_counts_baseline_window_1"]
swd_data = swd.data[:]
epoch_times = swd.starting_time + np.arange(len(swd_data)) / swd.rate
hours_swd = (epoch_times - bl1_start_time) / 3600

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(hours_swd, swd_data, lw=0.5)
axes[0].set_xlabel("Hours into baseline window 1")
axes[0].set_ylabel("SWD count / epoch")
axes[0].set_title(
    f"Spike-wave discharge counts per 5-s epoch - subject {nwb.subject.subject_id}"
)

swd = behavior["swd_epoch_counts_baseline_window_2"]
swd_data = swd.data[:]
epoch_times = swd.starting_time + np.arange(len(swd_data)) / swd.rate
hours_swd = (epoch_times - bl2_start_time) / 3600

axes[1].plot(hours_swd, swd_data, lw=0.5)
axes[1].set_xlabel("Hours into baseline window 2")
axes[1].set_ylabel("SWD count / epoch")

plt.tight_layout()
plt.show()

In [ ]:
# Always close the file when done
#io.close()